In [1]:
import sys
print(sys.executable)

/workspaces/llm-zoomcamp-2026/llm-zoomcamp-code/.venv/bin/python3


In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

83

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
import json
user_prompt = json.dumps(doc)

In [11]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
response.output_parsed.questions

['I just found this course — is it too late to join, or can I still sign up?',
 'Am I allowed to start the course after it has already begun?',
 'If I join now, can I still get the certificate somehow?',
 'What do I need to do to qualify for the certificate if I’m joining late?',
 'Is there still time to submit the project for a certificate, or did I miss the deadline?']

In [15]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [16]:
from evaluation_utils import llm_structured

In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still enroll and follow along, or is it too late?', 'Am I allowed to join the course after it’s already started?', 'If I start the course now, will I still be able to complete everything normally?', 'Can late joiners still participate in this course, or do I have to wait for the next run?', 'I missed the beginning of the course — can I still take it, and what do I need to do if I want a certificate?']


In [18]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=321)

In [19]:
from evaluation_utils import calc_price

In [20]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.000513, 'total_cost': 0.00066825}

In [21]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still enroll and follow along, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join the course after it’s already started?',
  'document': '74eb249bbf'},
 {'question': 'If I start the course now, will I still be able to complete everything normally?',
  'document': '74eb249bbf'},
 {'question': 'Can late joiners still participate in this course, or do I have to wait for the next run?',
  'document': '74eb249bbf'},
 {'question': 'I missed the beginning of the course — can I still take it, and what do I need to do if I want a certificate?',
  'document': '74eb249bbf'}]

In [22]:
import pandas as pd

In [23]:
pd.DataFrame(records)

,question,document
0,I just found this course — can I still enroll ...,74eb249bbf
1,Am I allowed to join the course after it’s alr...,74eb249bbf
2,"If I start the course now, will I still be abl...",74eb249bbf
3,Can late joiners still participate in this cou...,74eb249bbf
4,I missed the beginning of the course — can I s...,74eb249bbf


In [24]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'I just found this course a bit late — can I still start it now, and does that affect whether I can get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'Am I allowed to join the course after it has already started, or is it too late to enroll?',
   'document': '74eb249bbf'},
  {'question': 'If I begin the course now, will I still be able to earn the certificate, or is there a deadline I should know about?',
   'document': '74eb249bbf'},
  {'question': 'Is late registration okay for this course, and what do I need to do if I want the certificate?',
   'document': '74eb249bbf'},
  {'question': 'Can I still participate even though I discovered the course recently, and what’s required for certificate eligibility?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=132, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=339))

In [32]:
ground_truth = []
usages = []

for i in range(10):
    print("processing", i)
    records, usage = generate_ground_truth(documents[i])
    ground_truth.extend(records)
    usages.append(usage)
    print("done", i, len(records))

len(ground_truth)

processing 0
done 0 5
processing 1
done 1 5
processing 2
done 2 5
processing 3
done 3 5
processing 4
done 4 5
processing 5
done 5 5
processing 6
done 6 5
processing 7
done 7 5
processing 8
done 8 5
processing 9
done 9 5


50

In [33]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("ground_truth_data.csv", index=False)

In [34]:
df_ground_truth.head()

,question,document
0,Can I still join the course if I just found it...,74eb249bbf
1,Is it too late to sign up for the course and s...,74eb249bbf
2,"If I join late, can I still get a certificate ...",74eb249bbf
3,Do I have to finish the project before submiss...,74eb249bbf
4,What happens if I join now but miss the projec...,74eb249bbf


In [36]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [37]:
len(ground_truth)

25

In [38]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [43]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, documents[:10], generate_ground_truth)

  0%|          | 0/10 [00:00<?, ?it/s]

In [44]:
len(ground_truth)

25

In [45]:
print("hello")

hello


In [46]:
len(results)

10

In [29]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/83 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [47]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

50

In [48]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student, and where do I ask questions?',
 'document': '489dd1c9d9'}

In [49]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.006694499999999999

In [50]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.006694499999999999

In [53]:
results[1]

([{'question': 'I just signed up for LLM Zoomcamp — when should I expect the confirmation email, or do I actually need one?',
   'document': '977bf7786c'},
  {'question': 'Do I need to wait for any approval email before I can start the LLM Zoomcamp lessons and homework?',
   'document': '977bf7786c'},
  {'question': 'Is registration for the LLM Zoomcamp required to get access, or can I begin even if I never got a confirmation?',
   'document': '977bf7786c'},
  {'question': 'If I registered for the course but haven’t received anything back, does that mean my signup failed?',
   'document': '977bf7786c'},
  {'question': 'What’s the point of registering for LLM Zoomcamp if I can already start learning and submitting homework?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=129, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=367))

In [54]:
df_ground_truth = pd.DataFrame(ground_truth)

In [56]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [57]:
len(df_ground_truth)

50